# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations. For the iCGB21FR model, annotation of uniprot identifiers to the genes was made possible by using BLAST (work performed by Lorenzo Wormer, Karlsruhe institute of technology)

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [18]:
import pandas as pd
import os
from cobra.io import load_json_model, read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping, _parse_gpr
from Modules.utils.preprocessing import *

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_EnzymaticData_empty.xlsx')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_cglutamicumATCC13032_250124.xlsx')
NCBI_FILE_PATH = os.path.join('Data','Cglutanicum_phenotypes','NCBI_genes_corynglutatcc13032_250207.tsv')
GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_cgl_250124.json')
MODEL_FILE_PATH = os.path.join('Models', 'iCGB21FR_annotated_copyable.xml')


In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iCGB21FR_EnzymaticData_{formatted_date}.xlsx')

Relatively complicated regex because of complex gene names

1. WP_\d+_\d+: Matches identifiers starting with WP_ followed by digits (\d+), an underscore, and more digits.
2. |: Alternation operator to separate different patterns.
3. lcl_NC_\d+_\d+_prot_: Matches strings starting with lcl_NC_, followed by digits, an underscore, more digits, _prot_, and then:

    - WP_\d+_\d+_\d+: Matches WP_ identifiers with an additional _ and digits at the end.
    - |\d+: Matches a plain numeric sequence following _prot_.

4. |: Alternation operator for the next part.
5. cgl\d{4}: Matches strings starting with cgl followed by exactly 4 digits.

In [3]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'(?:WP_\d+_\d+|lcl_NC_\d+_\d+_prot_(?:WP_\d+_\d+_\d+|\d+)|[cC]gl\d{4}|[cC]g\d{4})'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BioModels](https://www.ebi.ac.uk/biomodels/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iCGB21FR model for [*Corynebacterium glutamicum* ATCC 13032](https://www.ebi.ac.uk/biomodels/MODEL2102050001#Overview)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `cgl` (*C. glutanicum* ATCC 13032) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Go to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *C. glutanicum* we used [this](https://www.uniprot.org/uniprotkb?query=(taxonomy_id:196627)) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

## 1. Get information from the model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [4]:
def create_gene_id_mapper_df(model):
    mapper_df = pd.DataFrame(columns = ['rxn_id', 'gene_id', 'uniprot_id', 'kegg_gene', 'NCBI_protein'])
    for rxn in model.reactions:
        for gene in rxn.genes:
            row = [rxn.id, gene.id]
            for dbxref in ['uniprot', 'kegg.genes','ncbiprotein']:
                if dbxref in gene.annotation:
                    row.append(gene.annotation[dbxref])
                else:
                    row.append(np.nan)
            mapper_df.loc[len(mapper_df)] = row
            
    #need to split of cgb: in front of kegg.gene id
    mapper_df['kegg_gene'] = mapper_df['kegg_gene'].apply(
    lambda gene: gene.split(':')[1] if isinstance(gene, str) and ':' in gene else gene
    )    
    print('Number of genes in the mapping dataframe: ',len(mapper_df))
    print('Number of genes mapped to KEGG identifier: ', len(mapper_df.kegg_gene.dropna()))
    print('Number of genes mapped to uniprot identifier: ', len(mapper_df.uniprot_id.dropna()))
    return mapper_df
            

print('mapping the ids from the iCGB21FR model')
model =read_sbml_model(MODEL_FILE_PATH)
id_mapper = create_id_mapper_from_model(model).rename({'ec-code': 'EC'}, axis=1)
print('\nmapping gene ids')
gene_id_mapper = create_gene_id_mapper_df(model)
id_mapper

mapping the ids from the iCGB21FR model
Mapped 532 of 1271 reactions to KEGG reaction IDs.

mapping gene ids
Number of genes in the mapping dataframe:  1659
Number of genes mapped to KEGG identifier:  1516
Number of genes mapped to uniprot identifier:  1595


,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN
2,12DGR161tipp,[nan],[nan],,False,NaN,NaN
3,12DGR180tipp,[nan],[nan],,False,NaN,NaN
4,12DGR181tipp,[nan],[nan],,False,NaN,NaN
...,...,...,...,...,...,...,...
1266,FRD7,"[[C00122, C00122;D02308], nan]","[C00042, nan]",,False,NaN,NaN
1267,GLYCLTt2rpp,"[C00160, C00080]","[C00160, C00080]",,True,NaN,NaN
1268,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN
1269,CYTB1,"[[C00007, C00007;D00003], C00080, nan]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",,False,NaN,NaN


### Map the NCBI gene names to KEGG gene names

In the iCGB21FR model, the gene names are by default the ncbi gene names. To map the gene-protein-reaction associations to the corresponding uniprot peptide identifiers, we need to map the model gene names to the kegg gene names.

In [5]:
gene_mapper = gene_id_mapper[['gene_id', 'kegg_gene']].set_index('gene_id').to_dict()['kegg_gene']
gene_mapper_ncbi_prot = gene_id_mapper.set_index('NCBI_protein').to_dict()['kegg_gene']
gene_mapper ={**{k:v for k, v in gene_mapper.items() if not pd.isna(v)}, 
              **{k:v for k, v in gene_mapper_ncbi_prot.items() if not pd.isna(v)}}

In [6]:
#replace the ncbi gene names in the GPR relation
def replace_ids(gpr, mapper):
    if pd.isna(gpr):  # Handle missing values
        return gpr
    for old_id, new_id in mapper.items():
        if pd.isna(new_id): continue
        gpr = gpr.replace(old_id, new_id)
    return gpr

id_mapper_gpr = id_mapper.copy()
# Apply the replacement function to the GPR column
id_mapper_gpr["GPR_old"] = id_mapper_gpr["GPR"].copy()
id_mapper_gpr["GPR"] = id_mapper_gpr["GPR"].apply(lambda x: replace_locustags_in_text(x, gene_mapper))
id_mapper_gpr["GPR_old"] = id_mapper_gpr["GPR"].apply(lambda x: replace_locustags_in_text(x, gene_mapper))

id_mapper_gpr

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,GPR_old
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN,
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN,
2,12DGR161tipp,[nan],[nan],,False,NaN,NaN,
3,12DGR180tipp,[nan],[nan],,False,NaN,NaN,
4,12DGR181tipp,[nan],[nan],,False,NaN,NaN,
...,...,...,...,...,...,...,...,...
1266,FRD7,"[[C00122, C00122;D02308], nan]","[C00042, nan]",,False,NaN,NaN,
1267,GLYCLTt2rpp,"[C00160, C00080]","[C00160, C00080]",,True,NaN,NaN,
1268,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN,WP_011014733_1
1269,CYTB1,"[[C00007, C00007;D00003], C00080, nan]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",,False,NaN,NaN,


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

Parse the BIGG gene-protein-reaction associations

In [7]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_locus_tags(text):
    if pd.isna(text):
        return text
    return re.findall(locustag_regex, text)

id_mapper_df = id_mapper_gpr.copy()

id_mapper_df['locus_tag'] = id_mapper_df['GPR'].apply(extract_locus_tags)

id_mapper_df['gene_id_old'] = id_mapper_df['GPR_old'].apply(extract_locus_tags)

id_mapper_df = id_mapper_df.explode(['locus_tag', 'gene_id_old'], ignore_index=True)

id_mapper_df = id_mapper_df.explode('EC', ignore_index=True)

id_mapper_df = id_mapper_df.merge(
    gene_id_mapper[['uniprot_id', 'gene_id']], left_on='gene_id_old', right_on='gene_id', how = 'left'
).drop(['gene_id_old', 'GPR_old'], axis=1)

id_mapper_df

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag,uniprot_id,gene_id
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN
2,12DGR161tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN
3,12DGR180tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN
4,12DGR181tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3056,GLYCLTt2rpp,"[C00160, C00080]","[C00160, C00080]",,True,NaN,NaN,NaN,NaN,NaN
3057,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN,WP_011014733_1,NaN,WP_011014733_1
3058,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN,WP_011014733_1,NaN,WP_011014733_1
3059,CYTB1,"[[C00007, C00007;D00003], C00080, nan]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",,False,NaN,NaN,NaN,NaN,NaN


In [8]:
def extract_kegg_gene_ids(text):
    if pd.isna(text):
        return text
    return re.findall(r'Cgl\d{4}', text)

# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['locus_tag'] = uniprot_df['Gene Names'].apply(extract_locus_tags)
# uniprot_df['kegg_gene'] = uniprot_df['Gene Names'].apply(extract_kegg_gene_ids)
uniprot_df = uniprot_df.explode('locus_tag', ignore_index=True)
# uniprot_df = uniprot_df.explode('kegg_gene', ignore_index=True)

uniprot_df = uniprot_df.drop(['Gene Names'], axis=1)
uniprot_df_genemapper = uniprot_df[['Entry', 'Mass', 'Length']]

### 2.3 Match Uniprot to BIGG data

In [9]:
# Merge first on locus_tag
id_mapper_with_protein = pd.merge(
    id_mapper_df, 
    uniprot_df_genemapper, 
    left_on='uniprot_id', 
    right_on = 'Entry',
    how='left'
)

id_mapper_with_protein = id_mapper_with_protein.drop('Entry', axis=1)
id_mapper_with_protein

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag,uniprot_id,gene_id,Mass,Length
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12DGR161tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12DGR180tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12DGR181tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3544,GLYCLTt2rpp,"[C00160, C00080]","[C00160, C00080]",,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3545,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN
3546,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN
3547,CYTB1,"[[C00007, C00007;D00003], C00080, nan]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Match the BiGG model reaction and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [10]:
id_mapper_final = id_mapper_with_protein.copy()
id_mapper_final = id_mapper_final.rename({'uniprot_id':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg.reaction', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,locus_tag,enzyme_id,gene_id,Mass,Length
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12DGR161tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12DGR180tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12DGR181tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,Cgl0023,R01664,3.1.3.5,C00239,5.4287
1,Cgl0023,R01664,3.1.3.89,C00239,5.4287
2,Cgl0023,R01664,3.1.3.-,C00239,5.4287
3,Cgl0023,R00511,3.1.3.5,C00475,5.9248
4,Cgl0023,R00511,3.1.3.91,C00475,5.9248


In [12]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = map_kcat_values_to_reaction_protein_association(id_mapper = id_mapper_final, 
                                                                     gotenzymes_df = eco_enzymes_df
                                                                    )
eco_enzymes_merged

Mapped 0 out of 3637 kcat values to reactions based on kegg gene id
Mapped 5855 out of 3637 kcat values to reactions based on ec number
Final merged dataset has 2956 unique reactions with kcat values.


,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,gene_id,Mass,Length,gene,ec_number,compound,kcat_values
0,12DGR120tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12DGR140tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12DGR161tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12DGR180tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12DGR181tipp,[nan],[nan],,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5850,GLYCLTt2rpp,"[C00160, C00080]","[C00160, C00080]",,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5851,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN,NaN,WP_011014733_1,NaN,NaN,WP_011014733_1,NaN,NaN,NaN
5852,PYRt,[C00022],[C00022],WP_011014733_1,False,NaN,NaN,NaN,WP_011014733_1,NaN,NaN,WP_011014733_1,NaN,NaN,NaN
5853,CYTB1,"[[C00007, C00007;D00003], C00080, nan]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [13]:

eco_enzymes_parsed = assign_directionalities_for_kcat_relations(eco_enzymes_merged.copy())
eco_enzymes_parsed = assign_defaults_for_proteins_without_mapping(eco_enzymes_parsed)
eco_enzymes_parsed

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,gene_id,molMass,Length,gene,kcat_values,direction
0,12DGR120tipp,[nan],[nan],Gene_12DGR120tipp,False,NaN,NaN,Enzyme_12DGR120tipp,NaN,39959.4825,325.0,Gene_12DGR120tipp,13.7,f
1,12DGR140tipp,[nan],[nan],Gene_12DGR140tipp,False,NaN,NaN,Enzyme_12DGR140tipp,NaN,39959.4825,325.0,Gene_12DGR140tipp,13.7,f
2,12DGR161tipp,[nan],[nan],Gene_12DGR161tipp,False,NaN,NaN,Enzyme_12DGR161tipp,NaN,39959.4825,325.0,Gene_12DGR161tipp,13.7,f
3,12DGR180tipp,[nan],[nan],Gene_12DGR180tipp,False,NaN,NaN,Enzyme_12DGR180tipp,NaN,39959.4825,325.0,Gene_12DGR180tipp,13.7,f
4,12DGR181tipp,[nan],[nan],Gene_12DGR181tipp,False,NaN,NaN,Enzyme_12DGR181tipp,NaN,39959.4825,325.0,Gene_12DGR181tipp,13.7,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6308,VOR2b,"[nan, C00010, C00141]","[C00630, nan, [C00011, C00011;D00004], C00080]",Gene_VOR2b,True,NaN,NaN,Enzyme_VOR2b,NaN,39959.4825,325.0,Gene_VOR2b,13.7,b
6309,VOR3b,"[nan, [C00671, C00671;C03465;C06008, C03465], ...","[[C01033, C01033;C15980, C15980], nan, [C00011...",Gene_VOR3b,True,NaN,NaN,Enzyme_VOR3b,NaN,39959.4825,325.0,Gene_VOR3b,13.7,b
6310,XYLI1,"[[C00181, C00181;D06346]]","[[C00310, C00310;C00508, C00309, C05052, C0835...",Gene_XYLI1_5.3.1.5,True,R01432,5.3.1.5,Enzyme_XYLI1_5.3.1.5,NaN,39959.4825,325.0,Gene_XYLI1_5.3.1.5,13.7,b
6311,XYLI2,"[[C00031, C00031;D00009, C00221, C00267]]","[[C00095, C01496, C05003, C10906, C00095;C0149...",Gene_XYLI2_5.3.1.5,True,R00307,5.3.1.5,Enzyme_XYLI2_5.3.1.5,NaN,39959.4825,325.0,Gene_XYLI2_5.3.1.5,13.7,b


In [15]:
eco_enzymes_mapped = eco_enzymes_parsed.copy()
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_mapped, model)
# eco_enzymes_mapped.gene = [[g] for g in eco_enzymes_mapped.gene]

gene2protein = {replace_ids(k, gene_mapper): v for k, v in gene2protein.items()}

# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_mapped, gene2protein)

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped

,rxn_id,Reactants,Products,GPR,reversible,kegg.reaction,EC,enzyme_id,gene_id,molMass,Length,gene,kcat_values,direction
0,12DGR120tipp,[nan],[nan],Gene_12DGR120tipp,False,NaN,NaN,Enzyme_12DGR120tipp,NaN,39959.4825,325.0,[Gene_12DGR120tipp],13.7,f
1,12DGR140tipp,[nan],[nan],Gene_12DGR140tipp,False,NaN,NaN,Enzyme_12DGR140tipp,NaN,39959.4825,325.0,[Gene_12DGR140tipp],13.7,f
2,12DGR161tipp,[nan],[nan],Gene_12DGR161tipp,False,NaN,NaN,Enzyme_12DGR161tipp,NaN,39959.4825,325.0,[Gene_12DGR161tipp],13.7,f
3,12DGR180tipp,[nan],[nan],Gene_12DGR180tipp,False,NaN,NaN,Enzyme_12DGR180tipp,NaN,39959.4825,325.0,[Gene_12DGR180tipp],13.7,f
4,12DGR181tipp,[nan],[nan],Gene_12DGR181tipp,False,NaN,NaN,Enzyme_12DGR181tipp,NaN,39959.4825,325.0,[Gene_12DGR181tipp],13.7,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5737,ZNabcpp,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,False,NaN,3.6.3.5,Enzyme_cg0042_Enzyme_cg2695,NaN,119878.4475,975.0,"[cg0042, cg2695]",13.7,f
5738,ZNabcpp,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,False,NaN,3.6.3.5,Enzyme_cg0042_Enzyme_cg0043,NaN,119878.4475,975.0,"[cg0042, cg0043]",13.7,f
5739,ZNabcpp,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,False,NaN,3.6.3.5,Enzyme_cg0042_Enzyme_cg0043,NaN,119878.4475,975.0,"[cg0042, cg0043]",13.7,f
5739,ZNabcpp,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,False,NaN,3.6.3.5,Enzyme_cg0042_Enzyme_cg2695,NaN,119878.4475,975.0,"[cg0042, cg2695]",13.7,f


In [16]:
#save the dataframe
#drop duplicate entries. If there are duplicates, make sure the mean kcat value is used to parametrize
eco_enzymes_final = eco_enzymes_mapped.groupby(
    ['rxn_id', 'enzyme_id', 'direction'], 
    as_index=False
).agg({
    'kcat_values': 'mean', 
    **{col: 'first' for col in eco_enzymes_mapped.columns if col not in ['rxn_id', 'enzyme_id', 'direction', 'kcat']
      }
})
# eco_enzymes_final.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [19]:
# Get the information about the enzyme sectors
other_sectors = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = None)
del other_sectors['ActiveEnzymes']

In [20]:
# Save it to a new excel file
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_final.to_excel(writer, sheet_name='ActiveEnzymes', index = False)
    for sheet, df in other_sectors.items():
        if sheet == 'ExcessEnzymes': sheet = 'UnusedEnzyme'
        df.to_excel(writer, sheet_name=sheet, index = False)